# 011.8 — Figure S10: betweenness centrality overview

**Pipeline step:** `011.8_FigS10_betweenness_overview` (network analysis series; follows `011.7_generate-network_simple-analysis` and uses the betweenness output of `011.4_network-modules_find-bottlennecks`).

**Reviewer request (R1.8):** *"Betweenness centrality is a very important concept in this paper. It is necessary to provide an overview plot of this metric, showing its distribution across all genes, as well as the biological significance of how different genes are distributed."*

**Input:** `analysis/network/betweenness_raw_all.csv` — betweenness centrality for all network nodes (representative pseudotime bin, Trajectory 3 / GEM+TGF-β1).

**What it produces:** a two-panel overview — (A) rank-ordered betweenness across every node, showing the heavy-tailed distribution with CDK1 and CDKN1A marked as outliers and all Fig. S11 molecular-gate proteins highlighted; (B) the top-ranked hubs by name, so the biological identity of the high-betweenness nodes is explicit.

**Outputs** (`figures_submission/supplementary/FigS10_betweenness_overview/`): 600 dpi PNG + vector PDF/SVG.

> The loader auto-detects the node-name and value columns and prints the CSV structure. If auto-detection is wrong, set `NAME_COL` / `VALUE_COL` explicitly in the config cell.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt




# ---- config --------------------------------------------------------------
if os.path.basename(os.getcwd()) == 'scripts_beta':
    os.chdir('../../')
print('working dir:', os.getcwd())

CSV     = 'analysis/network/betweenness_raw_all.csv'   # input file (relative to project root)
OUTDIR  = 'figures_submission/supplementary/FigS10_betweenness_overview'
HIGHLIGHT   = ['CDK1', 'CDKN1A']   # nodes to mark as outliers
TOP_LABEL_N = 12                   # how many top hubs to label in panel A
TOP_BAR_N   = 20                   # how many top hubs to show in panel B
NAME_COL    = None                 # None = auto-detect (node/gene column)
VALUE_COL   = None                 # None = auto-detect (betweenness column)

os.makedirs(OUTDIR, exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'svg.fonttype': 'none'})


In [ ]:
# ---- load + inspect ------------------------------------------------------
df = pd.read_csv(CSV)
print('shape:', df.shape)
print('columns:', list(df.columns))
display(df.head())

# auto-detect the node-name column (first non-numeric column, else the index)
if NAME_COL is None:
    obj_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if obj_cols:
        NAME_COL = obj_cols[0]
    else:
        df = df.reset_index().rename(columns={'index': 'node'})
        NAME_COL = 'node'

# auto-detect the betweenness value column
if VALUE_COL is None:
    num_cols = [c for c in df.columns if c != NAME_COL and pd.api.types.is_numeric_dtype(df[c])]
    betw = [c for c in num_cols if 'betw' in c.lower()]
    if betw:
        VALUE_COL = betw[0]
    elif len(num_cols) == 1:
        VALUE_COL = num_cols[0]
    elif len(num_cols) > 1:
        # multiple numeric columns (e.g. one per pseudotime bin): use the max across
        # bins as a representative value. To use a specific bin, set VALUE_COL above.
        df['betweenness_repr'] = df[num_cols].max(axis=1)
        VALUE_COL = 'betweenness_repr'
        print('\n[note] multiple numeric columns found; using row-max as representative.')
        print('       to use a specific Trajectory-3 bin, set VALUE_COL to that column name.')
    else:
        raise ValueError('No numeric column found for betweenness values.')

print(f'\nUsing NAME_COL = {NAME_COL!r},  VALUE_COL = {VALUE_COL!r}')

# tidy, rank, and locate the highlighted nodes
d = df[[NAME_COL, VALUE_COL]].dropna().copy()
d[NAME_COL] = d[NAME_COL].astype(str)
d = d.sort_values(VALUE_COL, ascending=False).reset_index(drop=True)
d['rank'] = np.arange(1, len(d) + 1)
up = {h.upper() for h in HIGHLIGHT}
d['hl'] = d[NAME_COL].str.upper().isin(up)
found = d.loc[d['hl'], NAME_COL].tolist()
missing = [h for h in HIGHLIGHT if h.upper() not in d[NAME_COL].str.upper().values]
print(f'nodes: {len(d)}  |  highlighted found: {found}  |  missing: {missing}')
for _, r in d[d['hl']].iterrows():
    print(f"  {r[NAME_COL]}: betweenness={r[VALUE_COL]:.4g}, rank {int(r['rank'])} of {len(d)}")


In [ ]:
# ---- figure --------------------------------------------------------------
# Molecular-gate proteins from Fig. S11 (TGF-β1 = gene symbol TGFB1)
GATE = ['CDK1', 'CDKN1A', 'WEE1', 'E2F1', 'MYBL2', 'CCNA2', 'SMAD3', 'TGFB1']
GATE_PRIMARY = {'CDK1': '#c0392b', 'CDKN1A': '#e67e22'}   # headline-labelled
GATE_COLOR   = '#138d75'                                   # teal for the other gate proteins
def hcolor(name):
    return GATE_PRIMARY.get(name.upper(), GATE_COLOR)

idx = {n.upper(): (int(rk), val) for n, rk, val in zip(d[NAME_COL], d['rank'], d[VALUE_COL])}
gate_found   = [g for g in GATE if g.upper() in idx]
gate_missing = [g for g in GATE if g.upper() not in idx]
print('gate found:  ', [(g, idx[g.upper()][0]) for g in gate_found])
print('gate missing:', gate_missing)   # if non-empty, gene absent from network or under another symbol

fig, (axA, axB) = plt.subplots(1, 2, figsize=(14, 6.2),
                               gridspec_kw={'width_ratios': [1.5, 1.0], 'wspace': 0.28})

# --- Panel A: rank-ordered betweenness across all nodes ---
axA.fill_between(d['rank'], d[VALUE_COL], color='#b8c4d0', alpha=0.55, lw=0, zorder=1)
axA.plot(d['rank'], d[VALUE_COL], color='#5d6d7e', lw=1.1, zorder=2)
axA.scatter(d['rank'], d[VALUE_COL], s=8, color='#95a5a6', zorder=2, rasterized=True)
xmax, ymax = d['rank'].max(), d[VALUE_COL].max()

# other gate members: teal dots (no inline labels, to avoid overlap)
for g in gate_found:
    if g.upper() in {k.upper() for k in GATE_PRIMARY}:
        continue
    rk, val = idx[g.upper()]
    axA.scatter(rk, val, s=75, color=GATE_COLOR, edgecolor='white', lw=1.1, zorder=5)

# CDK1 / CDKN1A: leader-line labels
offsets = {'CDK1': (0.14, 0.03), 'CDKN1A': (0.14, -0.03)}
for _, r in d[d['hl']].iterrows():
    c = hcolor(r[NAME_COL]); dx, dy = offsets.get(r[NAME_COL].upper(), (0.14, 0.0))
    axA.scatter(r['rank'], r[VALUE_COL], s=110, color=c, edgecolor='white', lw=1.4, zorder=6)
    axA.annotate(f"{r[NAME_COL]} (rank {int(r['rank'])})", xy=(r['rank'], r[VALUE_COL]),
                 xytext=(xmax*dx, r[VALUE_COL] + ymax*dy), fontsize=13, fontweight='bold',
                 color=c, va='center', arrowprops=dict(arrowstyle='-', color=c, lw=1.3))

# ranked list of all gate proteins, upper-right
x0, y0, dyy = 0.985, 0.97, 0.058
axA.text(x0, y0, 'Molecular-gate proteins', transform=axA.transAxes,
         ha='right', va='top', fontsize=10.5, fontweight='bold', color='#2c3e50')
for i, g in enumerate(GATE, start=1):
    txt = f'{g}: rank {idx[g.upper()][0]}' if g.upper() in idx else f'{g}: n/a'
    col = GATE_PRIMARY.get(g.upper(), GATE_COLOR if g.upper() in idx else '#95a5a6')
    bold = g.upper() in {k.upper() for k in GATE_PRIMARY}
    axA.text(x0, y0 - i*dyy, txt, transform=axA.transAxes, ha='right', va='top',
             fontsize=10, color=col, fontweight='bold' if bold else 'normal')

axA.set_xlabel('Node rank (by betweenness centrality)', fontsize=13)
axA.set_ylabel('Betweenness centrality', fontsize=13)
axA.set_title(f'A   Betweenness across all {len(d)} network nodes', fontsize=14, fontweight='bold', loc='left')
axA.tick_params(labelsize=11); axA.margins(x=0.02)
for s in ('top', 'right'):
    axA.spines[s].set_visible(False)

# --- Panel B: top hubs by name (gate proteins coloured) ---
def bcolor(name):
    u = name.upper()
    if u in {k.upper() for k in GATE_PRIMARY}:
        return [v for k, v in GATE_PRIMARY.items() if k.upper() == u][0]
    if u in {g.upper() for g in GATE}:
        return GATE_COLOR
    return '#8fa8bf'
top = d.head(TOP_BAR_N).iloc[::-1]
axB.barh(top[NAME_COL], top[VALUE_COL], color=[bcolor(n) for n in top[NAME_COL]], edgecolor='white', lw=0.5)
xpad = top[VALUE_COL].max() * 0.012
for y, (v, n) in enumerate(zip(top[VALUE_COL], top[NAME_COL])):
    axB.text(v + xpad, y, f'{v:.3g}', va='center', fontsize=9.5,
             fontweight='bold' if n.upper() in {g.upper() for g in GATE} else 'normal', color='#2c3e50')
axB.set_xlim(0, top[VALUE_COL].max() * 1.18)
axB.set_xlabel('Betweenness centrality', fontsize=13)
axB.set_title(f'B   Top {TOP_BAR_N} hub nodes', fontsize=14, fontweight='bold', loc='left')
for lbl in axB.get_yticklabels():
    if lbl.get_text().upper() in {g.upper() for g in GATE}:
        lbl.set_fontweight('bold'); lbl.set_color(bcolor(lbl.get_text()))
axB.tick_params(axis='y', labelsize=10.5); axB.tick_params(axis='x', labelsize=11)
for s in ('top', 'right'):
    axB.spines[s].set_visible(False)

fig.subplots_adjust(left=0.07, right=0.98, top=0.90, bottom=0.12)


In [ ]:
stem = os.path.join(OUTDIR, 'FigS10_betweenness_overview')
fig.savefig(stem + '.png', dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(stem + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(stem + '.svg', bbox_inches='tight', facecolor='white')
print('Saved 600 dpi PNG + vector PDF/SVG to:', OUTDIR)
plt.show()
